# 文档检索中的假设文档嵌入 (HyDE)

## 概述

此代码实现了用于文档检索的假设文档嵌入 (HyDE) 系统。 HyDE 是一种创新方法，它将查询问题转换为包含答案的假设文档，旨在弥合向量空间中查询和文档分布之间的差距。

## 动机

传统的检索方法经常会遇到短查询和更长、更详细的文档之间的语义差距。 HyDE 通过将查询扩展到完整的假设文档来解决这个问题，通过使查询表示与向量空间中的文档表示更相似来潜在地提高检索相关性。

## 关键组件

1. PDF processing and text chunking
2. 使用 FAISS 和 OpenAI 嵌入支持存储创建
3. 生成假设文档的语言模型
4. 实现 HyDE 技术的自定义 HyDE 搜索器类

## 方法详细信息

### 文档预处理和支持存储创建

1. The PDF is processed and split into chunks.
2. 使用 OpenAI 嵌入创建 FAISS 存储存储，以实现高效的相似性搜索。

### Hypothetical Document Generation

1. 语言模型 (GPT-4) 用于生成回答给定查询的假设文档。
2. 生成过程由提示模板引导，确保假设文档详细且与支持存储中使用的块大小相匹配。

### 检索过程

`HyDE检索器`类实现以下步骤：

1. 使用语言模型根据查询生成假设文档。
2. 使用假设文档作为支持存储中的搜索查询。
3. 检索与该假设文档最相似的文档。

## 主要特点

1. 查询扩展：将简短的查询转换为详细的假设文档。
2. 灵活的配置：允许调整块大小、重叠和检索文档的数量。
3. 与 OpenAI 模型集成：使用 GPT-4 进行假设文档生成，使用 OpenAI 嵌入进行矢量表示。

## 这种方法的好处

1. 提高相关性：通过将查询扩展到完整文档，HyDE 可以捕获更细微和相关的匹配。
2. 处理复杂查询：对于可能难以直接匹配的复杂或多方面查询特别有用。
3. 适应性：假设的文档生成可以适应不同类型的查询和文档领域。
4. 更好地理解上下文的潜力：扩展的查询可能会更好地捕获原始问题背后的上下文和意图。

## Implementation Details

1. 使用OpenAI的ChatGPT模型来生成假设文档。
2. 采用FAISS在向量空间中进行高效的相似性搜索。
3. 允许轻松可视化假设文档和检索结果。

## 结论

假设文档嵌入 (HyDE) 代表了一种创新的文档检索方法，解决了查询和文档之间的语义差距。通过利用高级语言模型将查询扩展到假设文档，HyDE 有可能显着提高检索相关性，特别是对于复杂或细致入微的查询。该技术在理解查询意图和上下文至关重要的领域（例如法律研究、学术文献综述或高级信息检索系统）特别有价值。

<div style="text-align: center;">

<img src="../images/HyDe.svg" alt="HyDe" style="width:40%; height:auto;">
</div>

<div style="text-align: center;">

<img src="../images/hyde-advantages.svg" alt="HyDe" style="width:100%; height:auto;">
</div>

# 包安装和导入

下面的单元格安装运行此笔记本所需的所有必需软件包。

In [ ]:
# Install required packages
!pip install python-dotenv

In [ ]:
# Clone the repository to access helper functions and evaluation modules
!git clone https://github.com/NirDiamant/RAG_TECHNIQUES.git
import sys
sys.path.append('RAG_TECHNIQUES')
# If you need to run with the latest data
# !cp -r RAG_TECHNIQUES/data .

In [ ]:
import os
import sys
from dotenv import load_dotenv


# Original path append replaced for Colab compatibility
from helper_functions import *
from evaluation.evalute_rag import *

# Load environment variables from a .env file
load_dotenv()

# Set the OpenAI API key environment variable
os.environ["OPENAI_API_KEY"] = os.getenv('OPENAI_API_KEY')

### 定义文档路径

In [ ]:
# Download required data files
import os
os.makedirs('data', exist_ok=True)

# Download the PDF document used in this notebook
!wget -O data/Understanding_Climate_Change.pdf https://raw.githubusercontent.com/NirDiamant/RAG_TECHNIQUES/main/data/Understanding_Climate_Change.pdf
!wget -O data/Understanding_Climate_Change.pdf https://raw.githubusercontent.com/NirDiamant/RAG_TECHNIQUES/main/data/Understanding_Climate_Change.pdf


In [2]:
path = "data/Understanding_Climate_Change.pdf"

### 定义 HyDe 搜索器类 - 创建支持存储、生成假设文档和检索

In [3]:
class HyDERetriever:
    def __init__(self, files_path, chunk_size=500, chunk_overlap=100):
        self.llm = ChatOpenAI(temperature=0, model_name="gpt-4o-mini", max_tokens=4000)

        self.embeddings = OpenAIEmbeddings()
        self.chunk_size = chunk_size
        self.chunk_overlap = chunk_overlap
        self.vectorstore = encode_pdf(files_path, chunk_size=self.chunk_size, chunk_overlap=self.chunk_overlap)
    
        
        self.hyde_prompt = PromptTemplate(
            input_variables=["query", "chunk_size"],
            template="""Given the question '{query}', generate a hypothetical document that directly answers this question. The document should be detailed and in-depth.
            the document size has be exactly {chunk_size} characters.""",
        )
        self.hyde_chain = self.hyde_prompt | self.llm

    def generate_hypothetical_document(self, query):
        input_variables = {"query": query, "chunk_size": self.chunk_size}
        return self.hyde_chain.invoke(input_variables).content

    def retrieve(self, query, k=3):
        hypothetical_doc = self.generate_hypothetical_document(query)
        similar_docs = self.vectorstore.similarity_search(hypothetical_doc, k=k)
        return similar_docs, hypothetical_doc


### 创建 HyDe 搜索器实例

In [4]:
retriever = HyDERetriever(path)

### 演示用例

In [5]:
test_query = "What is the main cause of climate change?"
results, hypothetical_doc = retriever.retrieve(test_query)

### 绘制假设文档和检索到的文档网

In [ ]:
docs_content = [doc.page_content for doc in results]

print("hypothetical_doc:\n")
print(text_wrap(hypothetical_doc)+"\n")
show_context(docs_content)

![](https://europe-west1-rag-techniques-views-tracker.cloudfunctions.net/rag-techniques-tracker?notebook=all-rag-techniques--hyde-hypothetical-document-embedding)